# REPLACE-BG User Data Generation

This notebook generates user data expansion for the REPLACE-BG dataset following the exact same pattern as DCLP3.csv structure.

## Dataset Overview
REPLACE-BG (Randomized Evaluation of Placement of CGM for All) is a study comparing:
- CGM Only: Continuous glucose monitoring only
- CGM+BGM: Continuous glucose monitoring plus blood glucose meter

The study evaluated different CGM monitoring strategies in adults with Type 1 diabetes using either Dexcom or Medtronic CGM systems.

In [ ]:
import pandas as pd
import numpy as np
import os

## Load REPLACE-BG Data Files

In [ ]:
# Define data paths
base_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/raw/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables'

# Load roster data
roster = pd.read_csv(os.path.join(base_path, 'HPtRoster.txt'), delimiter='|')
print(f"Roster shape: {roster.shape}")
print("Treatment groups:")
print(roster['TrtGroup'].value_counts())
roster.head()

In [ ]:
# Load screening data for demographic and device information
screening = pd.read_csv(os.path.join(base_path, 'HScreening.txt'), delimiter='|')
print(f"Screening shape: {screening.shape}")

# Load insulin delivery information
insulin_data = pd.read_csv(os.path.join(base_path, 'HInsulin.txt'), delimiter='|')
print(f"Insulin data shape: {insulin_data.shape}")
print("Insulin delivery routes:")
print(insulin_data['InsRoute'].value_counts(dropna=False))

# Load device information for insulin pumps
device_data = pd.read_csv(os.path.join(base_path, 'HDeviceUploads.txt'), delimiter='|')
insulin_pumps = device_data[device_data['DeviceType'].str.contains('insulin-pump', na=False)]
print(f"\\nInsulin pump device records: {len(insulin_pumps)}")
print("Pump models:")
pump_models = insulin_pumps.groupby('PtId')['DeviceModel'].first().value_counts()
print(pump_models.head(10))

# Load CGM information
print("\\nCGM devices from screening:")
print(screening['CGMUseDevice'].value_counts(dropna=False))

## Create User Data Expansion DataFrame

Following the exact same 10-column structure as DCLP3.csv:
1. id
2. insulin_delivery_device
3. insulin_delivery_algorithm
4. cgm_device
5. ethnicity
6. age_of_diagnosis
7. is_pregnant
8. insulin_delivery_modality
9. insulin_type_bolus
10. insulin_type_basal

In [ ]:
# Filter completed participants only
completed_roster = roster[roster['PtStatus'] == 'Completed'].copy()
print(f"Completed participants: {len(completed_roster)}")

# Merge roster with screening data
merged_data = completed_roster.merge(screening, on='PtID', how='inner')
print(f"Merged data shape: {merged_data.shape}")

In [ ]:
# Create user data expansion matching DCLP3 structure exactly
user_data_expansion = pd.DataFrame()

# 1. id
user_data_expansion['id'] = merged_data['PtID']

print(f"Created user data expansion with {len(user_data_expansion)} participants")

In [ ]:
# 2. insulin_delivery_device - Map from actual device data in REPLACE-BG
def map_insulin_delivery_device(pt_id):
    # Get device info for this patient
    patient_pumps = insulin_pumps[insulin_pumps['PtId'] == pt_id]
    
    if len(patient_pumps) == 0:
        return np.nan
    
    # Get the device model
    device_model = patient_pumps.iloc[0]['DeviceModel']
    
    if pd.isna(device_model):
        return np.nan
    
    device_str = str(device_model).strip()
    
    # Map device models to standardized names
    if 'OmniPod' in device_str:
        return 'OmniPod'
    elif 'MiniMed' in device_str or 'Paradigm' in device_str:
        return f'Medtronic {device_str}'
    elif 'Tandem' in device_str or 't:slim' in device_str:
        return device_str
    elif 'Animas' in device_str or 'Ping' in device_str or 'Vibe' in device_str:
        return f'Animas {device_str}' if 'Animas' not in device_str else device_str
    elif device_str.isdigit():  # Model numbers like 4628, 5448
        return f'Animas {device_str}'
    else:
        return device_str

user_data_expansion['insulin_delivery_device'] = merged_data['PtID'].apply(map_insulin_delivery_device)

print("Insulin delivery device distribution:")
print(user_data_expansion['insulin_delivery_device'].value_counts(dropna=False))

In [ ]:
# 3. insulin_delivery_algorithm - Map based on actual delivery route and device
def map_insulin_delivery_algorithm(pt_id, device):
    # Get insulin delivery route for this patient
    patient_insulin = insulin_data[insulin_data['PtID'] == pt_id]
    
    if len(patient_insulin) == 0:
        return np.nan
    
    ins_route = patient_insulin.iloc[0]['InsRoute']
    
    if ins_route == 'Pump':
        # For pumps in REPLACE-BG timeframe, most were basic pump therapy (SAP)
        # No advanced algorithms like Control-IQ were available yet
        return 'basal-bolus'  # Standard pump basal-bolus therapy
    elif ins_route == 'Injection':
        return 'basal-bolus'  # MDI basal-bolus therapy
    else:
        return np.nan

user_data_expansion['insulin_delivery_algorithm'] = merged_data.apply(
    lambda x: map_insulin_delivery_algorithm(x['PtID'], user_data_expansion.loc[user_data_expansion['id'] == x['PtID'], 'insulin_delivery_device'].iloc[0] if len(user_data_expansion.loc[user_data_expansion['id'] == x['PtID']]) > 0 else np.nan), axis=1
)

print("Insulin delivery algorithm distribution:")
print(user_data_expansion['insulin_delivery_algorithm'].value_counts(dropna=False))

In [ ]:
# 4. cgm_device - Map CGM devices for REPLACE-BG
def map_cgm_device(cgm_device):
    if pd.isna(cgm_device):
        return np.nan
    elif 'Dexcom' in str(cgm_device):
        return 'Dexcom G5'  # REPLACE-BG timeframe used G5
    elif 'Medtronic' in str(cgm_device):
        return 'Medtronic Guardian'
    else:
        return str(cgm_device)

user_data_expansion['cgm_device'] = merged_data['CGMUseDevice'].apply(map_cgm_device)

print("CGM device distribution:")
print(user_data_expansion['cgm_device'].value_counts(dropna=False))

In [ ]:
# 5. ethnicity - Map race and ethnicity for REPLACE-BG
def map_ethnicity_race(row):
    ethnicity = str(row['Ethnicity']) if pd.notna(row['Ethnicity']) else ''
    race = str(row['Race']) if pd.notna(row['Race']) else ''
    
    if ethnicity == 'Hispanic or Latino':
        if race == 'White':
            return 'White, Hispanic/Latino'
        else:
            return 'Hispanic/Latino'
    elif race == 'White':
        return 'White'
    elif race == 'Black or African American':
        return 'Black/African American'
    elif race == 'Asian':
        return 'Asian'
    elif race == 'American Indian or Alaska Native':
        return 'American Indian/Alaska Native'
    elif 'More than one race' in race:
        return 'More than one race'
    else:
        return race if race else 'Unknown'

user_data_expansion['ethnicity'] = merged_data.apply(map_ethnicity_race, axis=1)

print("Ethnicity distribution:")
print(user_data_expansion['ethnicity'].value_counts())

In [ ]:
# 6. age_of_diagnosis - REPLACE-BG has DiagAge
user_data_expansion['age_of_diagnosis'] = merged_data['DiagAge'].fillna(np.nan)

print("Age of diagnosis statistics:")
print(user_data_expansion['age_of_diagnosis'].describe())
print(f"Missing age of diagnosis: {user_data_expansion['age_of_diagnosis'].isna().sum()}")

In [ ]:
# 7. is_pregnant - Check if any pregnancy data available, otherwise default to False
# REPLACE-BG was an adult study, pregnancy status not specified in screening
user_data_expansion['is_pregnant'] = False

print("Is pregnant distribution:")
print(user_data_expansion['is_pregnant'].value_counts())

In [ ]:
# 8. insulin_delivery_modality - Map based on actual delivery route
def map_insulin_delivery_modality(pt_id):
    # Get insulin delivery route for this patient
    patient_insulin = insulin_data[insulin_data['PtID'] == pt_id]
    
    if len(patient_insulin) == 0:
        return np.nan
    
    ins_route = patient_insulin.iloc[0]['InsRoute']
    
    if ins_route == 'Pump':
        return 'SAP'  # Sensor-Augmented Pump (basic pump therapy)
    elif ins_route == 'Injection':
        return 'MDI'  # Multiple Daily Injections
    else:
        return np.nan

user_data_expansion['insulin_delivery_modality'] = merged_data['PtID'].apply(map_insulin_delivery_modality)

print("Insulin delivery modality distribution:")
print(user_data_expansion['insulin_delivery_modality'].value_counts(dropna=False))

In [ ]:
# 9. insulin_type_bolus - Map from actual insulin data
def get_insulin_type(pt_id, insulin_category='bolus'):
    patient_insulin = insulin_data[insulin_data['PtID'] == pt_id]
    
    if len(patient_insulin) == 0:
        return np.nan
    
    # Get all insulin names for this patient
    insulin_names = patient_insulin['InsName'].dropna().unique()
    
    if len(insulin_names) == 0:
        return np.nan
    
    if insulin_category == 'bolus':
        # Look for fast-acting insulins (bolus)
        for insulin in insulin_names:
            if any(fast_acting in str(insulin) for fast_acting in ['Humalog', 'Novolog', 'Fiasp', 'Apidra', 'Lispro', 'Aspart']):
                return insulin
        # If no specific fast-acting found, return the first insulin (for pump users this would be the pump insulin)
        return insulin_names[0]
    
    else:  # basal
        # For pump users, basal = bolus insulin
        delivery_route = patient_insulin.iloc[0]['InsRoute']
        if delivery_route == 'Pump':
            # For pumps, use the same insulin as bolus
            return get_insulin_type(pt_id, 'bolus')
        else:
            # For MDI users, look for long-acting insulins
            for insulin in insulin_names:
                if any(long_acting in str(insulin) for long_acting in ['Lantus', 'Levemir', 'Tresiba', 'Glargine', 'Detemir', 'Degludec']):
                    return insulin
            # If no long-acting found, return NaN (missing data)
            return np.nan

user_data_expansion['insulin_type_bolus'] = merged_data['PtID'].apply(lambda x: get_insulin_type(x, 'bolus'))

print("Insulin type bolus distribution:")
print(user_data_expansion['insulin_type_bolus'].value_counts(dropna=False))

In [ ]:
# 10. insulin_type_basal - Use same logic as bolus but for basal insulin
user_data_expansion['insulin_type_basal'] = merged_data['PtID'].apply(lambda x: get_insulin_type(x, 'basal'))

print("Insulin type basal distribution:")
print(user_data_expansion['insulin_type_basal'].value_counts(dropna=False))

In [ ]:
# Final dataframe summary
print("Final REPLACE-BG user data expansion:")
print(f"Shape: {user_data_expansion.shape}")
print("\nColumns:")
for col in user_data_expansion.columns:
    missing = user_data_expansion[col].isna().sum()
    print(f"{col}: {missing} missing values")

print("\nColumn structure matches DCLP3:")
expected_columns = ['id', 'insulin_delivery_device', 'insulin_delivery_algorithm', 'cgm_device', 'ethnicity', 'age_of_diagnosis', 'is_pregnant', 'insulin_delivery_modality', 'insulin_type_bolus', 'insulin_type_basal']
print(f"Expected: {expected_columns}")
print(f"Actual: {list(user_data_expansion.columns)}")
print(f"Match: {list(user_data_expansion.columns) == expected_columns}")

user_data_expansion.head(10)

## Save the Dataset

In [ ]:
# Save to the same folder as other datasets
output_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/ReplaceBG.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

user_data_expansion.to_csv(output_path, index=False)
print(f"REPLACE-BG user data expansion saved to: {output_path}")
print(f"Final dataset contains {len(user_data_expansion)} participants")